# 1. Setup
This notebook is meant for Save Version > Save & Run All (Commit). If a setup check fails, the notebook stops before any experiment runs.

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = "/root/.local/bin:/usr/local/bin:" + os.environ["PATH"]
os.environ["UV_PROJECT_ENVIRONMENT"] = "/tmp/dqn-venv"
!uv --version

In [ ]:
import os
if not os.path.exists("/kaggle/working/Deep-Q-Learning"):
    !git clone https://github.com/Ebi-Tech/Deep-Q-Learning.git /kaggle/working/Deep-Q-Learning
%cd /kaggle/working/Deep-Q-Learning
!git log -1 --oneline

In [ ]:
!uv sync

In [ ]:
%%bash
uv run python - <<'EOF'
import inspect
import os
import sys

import torch

import train

failures = 0
def check(name, condition):
    global failures
    print("ok    " if condition else "FAILED", name)
    if not condition:
        failures += 1

sys.argv = ["train.py", "--member", "Divine", "--experiment", "0"]
settings = train.get_settings()

check(f"current folder is the repo root: {os.getcwd()}",
      os.getcwd() == "/kaggle/working/Deep-Q-Learning" and os.path.isfile("train.py"))
check(f"CUDA is available: {torch.cuda.is_available()}", torch.cuda.is_available())
check(f"default train_every is 1: {settings.train_every}", settings.train_every == 1)
check(f"default total_steps is 1,000,000: {settings.total_steps:,}", settings.total_steps == 1_000_000)
check("QNetwork uses He (kaiming) initialisation", "kaiming_normal_" in inspect.getsource(train.QNetwork))
check("Adam uses eps=1.5e-4", "eps=1.5e-4" in inspect.getsource(train.main))

if failures:
    raise RuntimeError(f"{failures} setup check(s) FAILED, stopping before any experiment runs")
print("all checks passed")
EOF

In [ ]:
import csv
import os
import shutil

OUTPUT_DIR = "/kaggle/working/Divine_outputs"

def save_experiment(n):
    """Check experiment n, print its average reward, and copy its log and model into Divine_outputs.

    The log gets a row only when an episode ends, and the unfinished final episode is not logged,
    so the last logged step is always a little under 1,000,000. The 990,000 check only catches
    runs that were cut off. Files are copied right away so each finished experiment is kept even
    if a later one fails. A failed run is reported, never raised, so the remaining experiments still run.
    """
    log_path = f"logs/Divine/experiment_{n}.csv"
    model_path = f"models/Divine/experiment_{n}.pth"
    rows = []
    if os.path.isfile(log_path):
        with open(log_path) as f:
            rows = list(csv.DictReader(f))
    last_step = int(rows[-1]["step"]) if rows else 0
    checks = [
        (f"log {log_path} exists", os.path.isfile(log_path)),
        (f"last step is at least 990,000: {last_step:,}", last_step >= 990_000),
        (f"model {model_path} exists", os.path.isfile(model_path)),
    ]
    for name, passed in checks:
        print("ok    " if passed else "FAILED", name)
    if rows:
        last_100 = [float(row["reward"]) for row in rows[-100:]]
        print(f"average reward over the last {len(last_100)} episodes: {sum(last_100) / len(last_100):.2f}")
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    for path in (log_path, model_path):
        if os.path.isfile(path):
            shutil.copy(path, OUTPUT_DIR)
            print(f"copied {path} to {OUTPUT_DIR}")
    print("all checks passed" if all(passed for _, passed in checks) else "some checks FAILED")

# 2. Experiments
Every active cell below runs when you Save & Run All. Each run takes about 1 to 1.5 hours and a Kaggle session stops after 12 hours, so keep at most 4 experiments active per version. Comment out the ones already done.

Exp 3: learning rate 2.5e-4 (higher than baseline).

In [ ]:
# !uv run train.py --member Divine --experiment 3 --learning_rate 2.5e-4
# save_experiment(3)

Exp 4: learning rate 5e-4.

In [ ]:
# !uv run train.py --member Divine --experiment 4 --learning_rate 5e-4
# save_experiment(4)

Exp 5: learning rate 1e-3.

In [ ]:
# !uv run train.py --member Divine --experiment 5 --learning_rate 1e-3
# save_experiment(5)

Exp 6: gamma 0.95 (less weight on future rewards).

In [ ]:
# !uv run train.py --member Divine --experiment 6 --gamma 0.95
# save_experiment(6)

Exp 7: gamma 0.9 (next batch).

In [ ]:
!uv run train.py --member Divine --experiment 7 --gamma 0.9
save_experiment(7)

Exp 8: gamma 0.999, more weight on future rewards (next batch).

In [ ]:
!uv run train.py --member Divine --experiment 8 --gamma 0.999
save_experiment(8)

Exp 9: best learning rate + best gamma. Exp 10: second best learning rate + best gamma. Fill these in after experiments 2 to 8.

In [ ]:
# !uv run train.py --member Divine --experiment 9 --learning_rate BEST_LR --gamma BEST_GAMMA
# save_experiment(9)

In [ ]:
# !uv run train.py --member Divine --experiment 10 --learning_rate SECOND_BEST_LR --gamma BEST_GAMMA
# save_experiment(10)

# 3. Save everything
The zip and the Divine_outputs folder both appear in the version's Output tab.

In [ ]:
import shutil
import zipfile

os.makedirs("/kaggle/working/Divine_outputs", exist_ok=True)
shutil.make_archive("/kaggle/working/Divine_results", "zip", "/kaggle/working/Divine_outputs")
with zipfile.ZipFile("/kaggle/working/Divine_results.zip") as archive:
    names = sorted(archive.namelist())
print(f"/kaggle/working/Divine_results.zip contains {len(names)} file(s):")
for name in names:
    print("   ", name)